# Chapter 3 — qcirclab version

This notebook reproduces the examples of Chapter 3 using the lightweight `qcirclab` circuit library. The emphasis is on explicit circuit construction, simulation, measurement, composition, and canonical circuit patterns.


In [6]:
# In Google Colab, run this cell first.
!pip install -q git+https://github.com/2forts/qcirclab_repo.git


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
import numpy as np
from numpy import pi

from qcirclab import Circuit, bell_pair, ghz, teleportation
from qcirclab import gates as qg

np.set_printoptions(precision=3, suppress=True)


## Utility functions used in this notebook


In [8]:
def pretty_state(state, n_qubits=None, atol=1e-10):
    # Print nonzero amplitudes in computational-basis notation.
    state = np.asarray(state, dtype=complex).reshape(-1)
    if n_qubits is None:
        n_qubits = int(np.log2(state.size))
    terms = []
    for i, amp in enumerate(state):
        if abs(amp) > atol:
            terms.append(f"({amp:.3g})|{i:0{n_qubits}b}>")
    return " + ".join(terms) if terms else "0"


def circuit_unitary(qc: Circuit) -> np.ndarray:
    # Compute the full unitary matrix of a measurement-free circuit.
    n = qc.n_qubits
    U = np.zeros((2**n, 2**n), dtype=complex)
    for j in range(2**n):
        basis = np.zeros(2**n, dtype=complex)
        basis[j] = 1.0
        tmp = qc.copy().set_statevector(basis)
        U[:, j] = tmp.statevector()
    return U


def inverse_circuit(qc: Circuit, name="inverse") -> Circuit:
    # Construct the inverse of a measurement-free circuit by reversing gates.
    inv = Circuit(qc.n_qubits, qc.n_clbits, name=name)
    for op in reversed(qc.operations):
        if op.name in {"measure", "reset"}:
            raise ValueError("Cannot invert circuits with measurements or resets")
        if op.name == "barrier":
            continue
        inv.unitary(op.matrix.conj().T, op.targets, name=op.name + "dg", controls=op.controls)
    return inv


# Subsection 3.2.1 — Quantum and classical registers

`qcirclab` uses a compact register model: a circuit is created by specifying the total number of qubits and classical bits. Named registers can be represented pedagogically by index ranges.


In [9]:
# Equivalent to two quantum registers a[0:2], b[0:3], and one 2-bit classical register c
qreg_a = range(0, 2)   # qubits 0, 1
qreg_b = range(2, 5)   # qubits 2, 3, 4
creg_c = range(0, 2)   # classical bits 0, 1

qc = Circuit(5, 2)
print(qc.draw())


q0: |0>
q1: |0>
q2: |0>
q3: |0>
q4: |0>
c0:  0 
c1:  0 


In [10]:
# Measure the two-qubit register a into classical register c
for q, c in zip(qreg_a, creg_c):
    qc.measure(q, c)

print(qc.draw())


q0: |0>─M────
q1: |0>────M─
q2: |0>──────
q3: |0>──────
q4: |0>──────
c0:  0 ═╩════
c1:  0 ════╩═


# Subsection 3.2.2 — State initialization

Single-qubit states can be prepared using rotations, or arbitrary statevectors can be assigned directly when the goal is simulation and verification.


In [11]:
phi, theta, lam = pi/3, pi/4, -pi/2

qc = Circuit(1)
qc.rz(phi, 0)
qc.ry(theta, 0)
qc.rz(lam, 0)

print(qc.draw())
print("State:", pretty_state(qc.statevector()))


q0: |0>[ RZ][ RY][ RZ]
State: (0.892+0.239j)|0> + (0.099-0.37j)|1>


In [12]:
# Direct statevector initialization for simulation
state = np.array([1/np.sqrt(3), np.sqrt(2/3)], dtype=complex)
qc_init = Circuit(1).set_statevector(state)
print("Initialized state:", pretty_state(qc_init.statevector()))


Initialized state: (0.577+0j)|0> + (0.816+0j)|1>


In [13]:
# Basis-state initialization
qc_basis = Circuit(3).initialize_basis("101")
print("Basis state:", pretty_state(qc_basis.statevector()))


Basis state: (1+0j)|101>


# Subsection 3.2.3 — Special states


In [14]:
# Bell state
qc = Circuit(2)
qc.h(0)
qc.cx(0, 1)

print(qc.draw())
print("State:", pretty_state(qc.statevector(), 2))


q0: |0>[ H ]─●─
q1: |0>───  ─X─
State: (0.707+0j)|00> + (0.707+0j)|11>


In [15]:
# Uniform superposition on three qubits
qc = Circuit(3)
for qubit in range(3):
    qc.h(qubit)

print(qc.draw())
print("State:", pretty_state(qc.statevector(), 3))


q0: |0>[ H ]───  ───
q1: |0>───  [ H ]───
q2: |0>───  ───  [ H ]
State: (0.354+0j)|000> + (0.354+0j)|001> + (0.354+0j)|010> + (0.354+0j)|011> + (0.354+0j)|100> + (0.354+0j)|101> + (0.354+0j)|110> + (0.354+0j)|111>


In [16]:
# GHZ state
qc = Circuit(3)
qc.h(0)
for i in range(1, 3):
    qc.cx(0, i)

print(qc.draw())
print("State:", pretty_state(qc.statevector(), 3))


q0: |0>[ H ]─●──●─
q1: |0>───  ─X────
q2: |0>───  ────X─
State: (0.707+0j)|000> + (0.707+0j)|111>


# Subsection 3.3.1 — Circuit composition


In [17]:
qc1 = Circuit(1)
qc1.h(0)

qc2 = Circuit(1)
qc2.s(0)

qc_combined = qc1.copy().compose(qc2)  # sequential: qc1 followed by qc2
print(qc_combined.draw())


q0: |0>[ H ][ S ]


In [18]:
# Parallel composition can be represented by composing subcircuits onto disjoint qubits.
qc_parallel = Circuit(2)
qc_a = Circuit(1).h(0)
qc_b = Circuit(1).x(0)

qc_parallel.compose(qc_a, qubits=[0])
qc_parallel.compose(qc_b, qubits=[1])

print(qc_parallel.draw())


q0: |0>[ H ]───
q1: |0>───  [ X ]


# Subsection 3.3.2 — Subcircuits and reusability


In [19]:
# Define a reusable two-qubit Bell subcircuit
bell = Circuit(2, name="bell")
bell.h(0)
bell.cx(0, 1)

# Append the same subcircuit on two disjoint pairs of qubits
qc = Circuit(4)
qc.append(bell, qubits=[0, 1])
qc.append(bell, qubits=[2, 3])

print(qc.draw())
print("State:", pretty_state(qc.statevector(), 4))


q0: |0>[ H ]─●────  ───
q1: |0>───  ─X────  ───
q2: |0>───  ───[ H ]─●─
q3: |0>───  ──────  ─X─
State: (0.5+0j)|0000> + (0.5+0j)|0011> + (0.5+0j)|1100> + (0.5+0j)|1111>


In [20]:
# Append a subcircuit followed by its inverse
inv_bell = inverse_circuit(bell)

qc2 = Circuit(2)
qc2.append(bell, qubits=[0, 1])
qc2.append(inv_bell, qubits=[0, 1])

print(qc2.draw())
print("Final state:", pretty_state(qc2.statevector(), 2))
print("Unitary close to identity:", np.allclose(circuit_unitary(qc2), np.eye(4)))


q0: |0>[ H ]─●──●─  [HDG]
q1: |0>───  ─X─[CXD]───
Final state: (1+0j)|00>
Unitary close to identity: True


# Subsection 3.3.3 — Parameterized circuits

The lightweight library stores numerical gates, so symbolic parameters are naturally represented by Python functions that build circuits once parameters are provided.


In [21]:
def rotation_ansatz(theta, phi):
    qc = Circuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return qc

qc = rotation_ansatz(theta=1.2, phi=-0.5)
print(qc.draw())
print("State:", pretty_state(qc.statevector()))


q0: |0>[ RY][ RZ]
State: (0.8+0.204j)|0> + (0.547-0.14j)|1>


In [22]:
# Reusing the same ansatz with different parameter values
for theta, phi in [(0.2, 0.0), (1.2, -0.5), (2.0, 1.0)]:
    qc = rotation_ansatz(theta, phi)
    print(f"theta={theta}, phi={phi}")
    print(pretty_state(qc.statevector()))


theta=0.2, phi=0.0
(0.995+0j)|0> + (0.0998+0j)|1>
theta=1.2, phi=-0.5
(0.8+0.204j)|0> + (0.547-0.14j)|1>
theta=2.0, phi=1.0
(0.474-0.259j)|0> + (0.738+0.403j)|1>


# Subsection 3.4.1 — Measurement in the computational basis


In [23]:
qc = Circuit(1, 1)
qc.h(0)
qc.measure(0, 0)

print(qc.draw())
result = qc.run(shots=1024, seed=7)
print(result.counts)


q0: |0>[ H ]─M─
c0:  0      ═╩═
{'0': 507, '1': 517}


In [24]:
qc = Circuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure(0, 0)
qc.measure(1, 1)

print(qc.draw())
result = qc.run(shots=1024, seed=7)
print(result.counts)


q0: |0>[ H ]─●──M────
q1: |0>───  ─X─────M─
c0:  0         ═╩════
c1:  0         ════╩═
{'00': 500, '11': 524}


# Subsection 3.4.2 — Mid-circuit measurement


In [25]:
qc = Circuit(2, 1)
qc.h(0)
qc.cx(0, 1)
qc.measure(0, 0)
qc.z(1)

print(qc.draw())
result = qc.run(shots=10, seed=3)
print("Memory:", result.memory)
print("Final classical register from last shot:", result.final_classical)
print("Final state from last shot:", pretty_state(result.final_state, 2))


q0: |0>[ H ]─●──M────
q1: |0>───  ─X────[ Z ]
c0:  0         ═╩═   
Memory: ['1', '1', '0', '0', '1', '1', '1', '1', '0', '1']
Final classical register from last shot: (1,)
Final state from last shot: (-1+0j)|11>


# Subsection 3.4.3 — Classical conditional operations


In [26]:
qc = Circuit(2, 1)
qc.h(0)
qc.measure(0, 0)

# Apply X to qubit 1 only if classical bit 0 is 1
qc.x(1, condition=(0, 1))

print(qc.draw())
result = qc.run(shots=20, seed=4)
print(result.counts)
print("Example memories:", result.memory[:10])


q0: |0>[ H ]─M────
q1: |0>───  ───[ X ]
c0:  0      ═╩═   
{'0': 13, '1': 7}
Example memories: ['0', '0', '0', '1', '0', '1', '0', '1', '0', '0']


# Subsection 3.5.1 — Simulation backends


In [35]:
# Define a simple Bell circuit without measurement
qc_state = Circuit(2)
qc_state.h(0)
qc_state.cx(0, 1)

state = qc_state.statevector()
print("Statevector:", state)
print("State:", pretty_state(state, 2))

# Shot-based simulation requires measurement operations
qc_meas = Circuit(2, 2)
qc_meas.h(0)
qc_meas.cx(0, 1)
qc_meas.measure(0, 0)
qc_meas.measure(1, 1)

result = qc_meas.run(shots=1024, seed=9)
print("Counts:", result.counts)
print("Probabilities:", result.probabilities())


Statevector: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]
State: (0.707+0j)|00> + (0.707+0j)|11>
Counts: {'00': 492, '11': 532}
Probabilities: {'00': 0.48046875, '11': 0.51953125}


# Subsection 3.5.2 — Inspecting intermediate states


In [36]:
qc = Circuit(2)
qc.h(0)
state1 = qc.statevector()
print("After H on qubit 0:", pretty_state(state1, 2))

qc.cx(0, 1)
state2 = qc.statevector()
print("After CNOT:", pretty_state(state2, 2))


After H on qubit 0: (0.707+0j)|00> + (0.707+0j)|10>
After CNOT: (0.707+0j)|00> + (0.707+0j)|11>


In [37]:
# The run() method also stores snapshots for the last shot.
qc = Circuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure(0, 0)
qc.measure(1, 1)

result = qc.run(shots=1, seed=5)
for op_name, state, classical in result.snapshots:
    print(op_name, "classical=", classical, "state=", pretty_state(state, 2))

h classical= (0, 0) state= (0.707+0j)|00> + (0.707+0j)|10>
cx classical= (0, 0) state= (0.707+0j)|00> + (0.707+0j)|11>
measure classical= (0, 0) state= (1+0j)|00>
measure classical= (0, 0) state= (1+0j)|00>


# Subsection 3.6.1 — Bell pair circuit


In [31]:
qc = Circuit(2)
qc.h(0)
qc.cx(0, 1)
print(qc.draw())
print(pretty_state(qc.statevector(), 2))


q0: |0>[ H ]─●─
q1: |0>───  ─X─
(0.707+0j)|00> + (0.707+0j)|11>


# Subsection 3.6.2 — GHZ state circuit


In [32]:
n = 3
qc = Circuit(n)
qc.h(0)
for i in range(1, n):
    qc.cx(0, i)

print(qc.draw())
print(pretty_state(qc.statevector(), n))


q0: |0>[ H ]─●──●─
q1: |0>───  ─X────
q2: |0>───  ────X─
(0.707+0j)|000> + (0.707+0j)|111>


# Subsection 3.6.3 — SWAP test circuit

A controlled-SWAP can be decomposed using Toffoli gates. The example below prepares two simple target states and measures the ancilla.


In [33]:
def cswap(qc, control, q1, q2):
    qc.ccx(control, q2, q1)
    qc.ccx(control, q1, q2)
    qc.ccx(control, q2, q1)
    return qc

qc = Circuit(3, 1)

# Ancilla qubit 0; target states on qubits 1 and 2.
# Example: prepare |+> on qubit 1 and |0> on qubit 2.
qc.h(1)

qc.h(0)
cswap(qc, 0, 1, 2)
qc.h(0)
qc.measure(0, 0)

print(qc.draw())
result = qc.run(shots=1024, seed=11)
print(result.counts)
print(result.probabilities())


q0: |0>───  [ H ]─●─  ─●─  ─●─  [ H ]─M─
q1: |0>[ H ]───  [CCX]─●─  [CCX]───  ───
q2: |0>───  ───  ─●─  [CCX]─●─  ───  ───
c0:  0                               ═╩═
{'1': 265, '0': 759}
{'1': 0.2587890625, '0': 0.7412109375}


# Subsection 3.6.4 — Quantum teleportation circuit


In [34]:
qc = Circuit(3, 2)

# Optional: prepare a nontrivial state on qubit 0 to be teleported.
qc.ry(np.pi/5, 0)
qc.rz(np.pi/7, 0)

# Step 1: Create a Bell pair between qubits 1 and 2
qc.h(1)
qc.cx(1, 2)

# Step 2: Entangle qubit 0 with qubit 1
qc.cx(0, 1)
qc.h(0)

# Step 3: Measure the first two qubits
qc.measure(0, 0)
qc.measure(1, 1)

# Step 4: Apply conditional corrections to qubit 2
qc.x(2, condition=(1, 1))
qc.z(2, condition=(0, 1))

print(qc.draw())
result = qc.run(shots=5, seed=12)
print("Memory:", result.memory)
print("Final classical register:", result.final_classical)
print("Final state from last shot:", pretty_state(result.final_state, 3))


q0: |0>[ RY][ RZ]───  ────●─[ H ]─M───────  ───
q1: |0>───  ───  [ H ]─●──X────  ────M────  ───
q2: |0>───  ───  ───  ─X───────  ──────[ X ][ Z ]
c0:  0                           ═╩════        
c1:  0                           ════╩═        
Memory: ['10', '11', '11', '01', '00']
Final classical register: (0, 0)
Final state from last shot: (0.927-0.212j)|000> + (0.301+0.0688j)|001>
